# Ch.3　データを「学ばせる」── 機械学習で品種を予測する

| 項目 | 内容 |
|------|------|
| **使うデータ** | ワインデータ（Ch.1・2 と同じ）178件・13特徴量・3品種 |
| **作るもの** | 品種予測モデル ＋ 正解率 ＋ 混同行列 ＋ 特徴量重要度グラフ |
| **実務での対応物** | 需要予測・品質管理・顧客分類 |
| **演習時間** | 35分（問3まで完了で合格） |

> 🔑 **今日のゴール：** `fit → predict` という型を体に染み込ませる。  
> **この4ステップはどんな機械学習でも共通です。**
> ```
> ① データを準備する（X と y に分ける）
> ② 訓練/テストに分割する
> ③ モデルを学習させる（fit）
> ④ 予測・評価する（predict → accuracy）
> ```

---
## 🤖 Copilot の使い方

**URL：** https://copilot.microsoft.com

**質問の型（5点を揃える）：**
```
【やりたいこと】〇〇したい
【使うライブラリ】scikit-learn
【データの形】X は 178行×13列の DataFrame、y は品種名の Series
【環境】Python 3.8.6、Windows、JupyterLab
【困っていること】〇〇の書き方がわからない
```

---
## 準備：ライブラリとデータを読み込む

> ⚠️ **まず最初にこのセルを実行してください。**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種名"] = [wine.target_names[t] for t in wine.target]

print(f"✅ 読み込み完了　行数: {len(df)}  列数: {len(df.columns)}")

---
## データを読み込む

> ℹ️ **このセルは実行するだけでOKです。**  
> ワインデータを読み込み、品種名の列を追加した DataFrame を作ります。

In [ ]:
# ワインデータを読み込んで DataFrame に変換する（実行するだけでOK）

**✅ エラーなく表示されれば OK です。**  
> `import` でエラーが出た場合は Copilot に「ImportError の直し方を教えて」と聞いてください。

---
## STEP 1　データを「X（特徴量）」と「y（正解ラベル）」に分ける

### ❓ なぜ X と y に分けるのか

> **機械学習は「X（入力）を見て y（答え）を予測する」仕組みだからです。**

| 変数 | 意味 | 今回の中身 |
|------|------|----------|
| **X** | 入力データ（特徴量） | アルコール度数・プロリンなど 13列 |
| **y** | 正解データ（ラベル） | 品種名（wine_0 / wine_1 / wine_2） |

> 📌 Ch.2 で「この変数が品種の決め手になりそう」と仮説を立てましたね。  
> X にその変数が含まれています。モデルが重要と判断するか、後で確認します。

---
### 📝 タスク 1-1　X（特徴量）を作ってください

**何をするか：** `品種名` 列を除いた DataFrame を X とする  
**使う方法：** `df.drop(columns=["除く列名"])`

In [ ]:
# X：品種名以外の全列を特徴量として使う

---
### 📝 タスク 1-2　y（正解ラベル）を作ってください

**何をするか：** `品種名` 列だけを取り出して y とする

In [ ]:
# y：予測したい品種名の列

---
## STEP 2　データを「訓練用」と「テスト用」に分ける

### ❓ なぜ訓練とテストに分けるのか

> **「試験の答えを見て勉強した人」が高得点を取っても意味がないからです。**  
> テストデータはモデルが「初めて見るデータ」として使います。

```
全データ（178件）
├── 訓練データ（約125件）→ モデルが勉強するデータ
└── テストデータ（約53件）→ モデルを採点するデータ（学習には使わない！）
```

---
### 📝 タスク 2-1　データを 7:3 に分割してください

**何をするか：** `train_test_split(X, y, test_size=0.3, random_state=42)` を実行する  
**`random_state=42` とは：** 毎回同じ分け方になるおまじない（再現性のため）

In [ ]:
# データを訓練用とテスト用に分割する

**✅ チェックポイント**
- [ ] 訓練データ＋テストデータ = 178件になっているか
- [ ] テストデータが全体の約 30% になっているか

---
## 問1　モデルを学習させて予測する ★☆☆

### ❓ なぜ RandomForest を使うのか

> **「多数決」で答えを出すため、精度が安定しているからです。**  
> 決定木（Yes/No の分岐）を 100本作り、その多数決で品種を決めます。

```
「alcohol > 13 か？」
        ↓
    Yes → Barolo？      ← 1本の決定木
    No  → Barbera？

100本の多数決 → 最も票が多い品種を予測結果とする ← RandomForest
```

---
### 📝 タスク 1-1　RandomForest モデルを作ってください

**何をするか：** `RandomForestClassifier()` でモデルを作る（木100本・再現性固定）

In [ ]:
# RandomForest モデルを作る

---
### 📝 タスク 1-2　訓練データでモデルを学習させてください

**何をするか：** `model.fit(訓練X, 訓練y)` を実行する  
**意味：** 「X を見て y を当てる方法」を 100本の木が学習する

In [ ]:
# 訓練データで学習する（fit）

---
### 📝 タスク 1-3　テストデータで予測してください

**何をするか：** `model.predict(テストX)` を実行する  
**意味：** モデルが「初めて見る」テストデータの品種を予測する

In [ ]:
# テストデータで品種を予測する（predict）

**✅ チェックポイント**
- [ ] `model.fit()` がエラーなく完了したか
- [ ] `y_pred` に品種名（wine_0 / wine_1 / wine_2）が入っているか

---
## 問2　正解率でモデルを評価する ★★☆

### ❓ なぜ正解率を使うのか

> **「モデルが何割正解したか」が最も直感的な評価指標だからです。**  
> 今回のゴールは **正解率 90% 以上** です。

```
正解率 = 正解した件数 ÷ テストデータの総件数
```

---
### 📝 タスク 2-1　正解率を計算してください

**何をするか：** `accuracy_score(正解, 予測)` を実行する

In [ ]:
# 正解率を計算する

---
### 📝 タスク 2-2　正解率が 90% 以上かを確認してください

**何をするか：** 正解率の数値を見て判断する  
**Copilot に聞く場合：** 「正解率が 90% 未満だった場合に試せる改善方法を教えて」

In [ ]:
# 90% 以上かを判定する

**✅ チェックポイント**
- [ ] 正解率が表示されているか
- [ ] 90% 以上か（RandomForest は高精度なので多くの場合達成できる）

---
## 問3　混同行列で「どの品種を間違えたか」を調べる ★★☆（最重要）

### ❓ なぜ混同行列を使うのか

> **正解率だけでは「どの品種を間違えたか」がわかりません。**  
> 混同行列は「正解 vs 予測」の対応表で、間違いのパターンを把握できます。

```
              予測: wine_0  予測: wine_1  予測: wine_2
正解: wine_0 │    18    │     0    │     0    │ ← 全問正解
正解: wine_1 │     0    │    19    │     2    │ ← 2件を wine_2 と間違えた
正解: wine_2 │     0    │     0    │    14    │ ← 全問正解

対角線 = 正解　対角線以外 = 間違い
```

---
### 📝 タスク 3-1　混同行列を計算してください

**何をするか：** `confusion_matrix(正解, 予測)` を実行する

In [ ]:
# 混同行列を計算する

---
### 📝 タスク 3-2　混同行列をヒートマップで可視化してください

**何をするか：** `sns.heatmap()` で見やすく表示する  
**Copilot に聞く場合：** 「seaborn の heatmap で混同行列を可視化する書き方を教えて。annot=True で数値を表示したい」

In [ ]:
# 混同行列をヒートマップで表示する

---
### 📝 タスク 3-3　混同行列から気づいたことを書いてください（AI 禁止）

> ⛔ **この考察は自分で書いてください。AI は使わないこと。**

> 📌 **なぜ大事なのか：**  
> 「どの品種をどの品種と間違えているか」を読み取る力が  
> データサイエンティストの本質的なスキルです。

---

**【発見】対角線の外（間違い）に数字はありましたか？**

>

**【解釈】どの品種をどの品種と間違えていましたか？**

>

**【Ch.2 との答え合わせ】Ch.2 で「効く変数」と予測した変数は重要でしたか？**

>

**✅ チェックポイント**
- [ ] 混同行列のヒートマップが表示されているか
- [ ] 対角線（正解）が濃い色になっているか
- [ ] 間違いのパターンを言葉で説明できるか

---
## 問4（発展）　特徴量重要度で「効いた変数」を確認する ★★★

> ⏰ **時間が余った人だけ取り組んでください**

### ❓ なぜ特徴量重要度を見るのか

> **「モデルが品種を判断するときに最も使った変数」がわかります。**  
> Ch.2 で立てた仮説（「この変数が効くはず」）と答え合わせができます。

---
### 📝 タスク 4-1　特徴量重要度を取り出してください

**何をするか：** `model.feature_importances_` を DataFrame にまとめる

In [ ]:
# 特徴量重要度を取り出す

---
### 📝 タスク 4-2　特徴量重要度を横棒グラフで可視化してください

**何をするか：** `importances.plot(kind="barh")` で横棒グラフを描く  
**Copilot に聞く場合：** 「pandas の Series を横棒グラフ（barh）で表示する方法を教えて」

In [ ]:
# 特徴量重要度を横棒グラフで表示する

**✅ チェックポイント**
- [ ] 横棒グラフが表示されているか
- [ ] 上位3位の変数はどれか
- [ ] Ch.2 で仮説を立てた変数は上位に入っていたか